<a name="title"></a>
# Live-Learning Research Agent with Perplexity (Search + Embeddings + Agent) and Qdrant

*Notebook by the [Perplexity](https://perplexity.ai) API team.*

This cookbook builds a **single research agent that uses all three Perplexity APIs together** through the [`perplexity-haystack`](https://haystack.deepset.ai/integrations/perplexity) integration:

| Perplexity API | Haystack component | Role in this notebook |
|---|---|---|
| **Agent API** (`POST /v1/agent`) | `PerplexityChatGenerator` | The agent's reasoning model. OpenAI-Responses-compatible, so it slots into Haystack's [`Agent`](https://docs.haystack.deepset.ai/docs/agent) with no glue. |
| **Search API** (`POST /search`) | `PerplexityWebSearch` | Ranked, cleaned, cited web results, exposed to the agent as a tool — replaces the SerperDev / DuckDuckGo + `LinkContentFetcher` chain other cookbooks build by hand. |
| **Embeddings API** (`POST /v1/embeddings`) | `pplx-embed-v1-0.6b` (called directly, see note below) | Indexes documents into [Qdrant](https://haystack.deepset.ai/integrations/qdrant-document-store) and embeds queries at retrieval time. |

The agent gets three tools — `retrieve_from_index`, `web_search`, `ingest_url` — and decides per question whether to read the local index, search the live web, or grow the index with a freshly-fetched page. Net result: a knowledge base that *learns from the agent's own behaviour*, with citations on every web answer.

> **One real-world gotcha you'll hit:** the live `/v1/embeddings` endpoint only accepts `encoding_format` of `base64_int8` or `base64_binary`, but `PerplexityDocumentEmbedder` / `PerplexityTextEmbedder` inherit the OpenAI default of `float` and get back HTTP 400. Until that's fixed upstream, we call the embeddings endpoint directly with `httpx` and decode `base64_int8` to `float32`. It's ~15 lines and we walk through it below.


<a name="what-you-will-build"></a>
## What you will build

1. A Qdrant Cloud–backed knowledge base seeded with a couple of Haystack documentation snippets, embedded with `pplx-embed-v1-0.6b`.
2. A `web_search` tool wrapping `PerplexityWebSearch` so the agent can hit the Perplexity Search API directly.
3. An `ingest_url` tool that takes a URL from a web search result, extracts the page with `trafilatura`, embeds the chunks with the Perplexity Embeddings API, and writes them to Qdrant.
4. A `retrieve_from_index` tool that embeds the query and pulls the top-k from Qdrant.
5. A Haystack [`Agent`](https://docs.haystack.deepset.ai/docs/agent) driven by `PerplexityChatGenerator` that orchestrates the three tools.
6. Three sample questions that show the index growing across turns and answers carrying citations end-to-end.


<a name="setup"></a>
## 1. Setup

Install the integration packages plus `trafilatura` for HTML extraction. The notebook uses a **Qdrant Cloud** cluster (free tier is enough) so the index persists across runs — flip `recreate_index=True` to `False` once you have something you want to keep.


In [ ]:
%pip install -q \
    "haystack-ai>=2.24.1" \
    "perplexity-haystack" \
    "qdrant-haystack" \
    "trafilatura" \
    "httpx" \
    "numpy"

<a name="credentials"></a>
### 1.1 Credentials

You'll need two things:

* A **Perplexity API key** from [https://www.perplexity.ai/account/api](https://www.perplexity.ai/account/api).
* A **Qdrant Cloud cluster URL + API key** from [https://cloud.qdrant.io](https://cloud.qdrant.io) (free tier works fine).

The cell below uses `getpass` so the keys never end up in the notebook output.


In [ ]:
import os
from getpass import getpass

if not os.environ.get("PERPLEXITY_API_KEY"):
    os.environ["PERPLEXITY_API_KEY"] = getpass("PERPLEXITY_API_KEY: ")
if not os.environ.get("QDRANT_URL"):
    os.environ["QDRANT_URL"] = getpass("QDRANT_URL (e.g. https://xxxx.cloud.qdrant.io): ")
if not os.environ.get("QDRANT_API_KEY"):
    os.environ["QDRANT_API_KEY"] = getpass("QDRANT_API_KEY: ")

print("Perplexity key set:", bool(os.environ.get("PERPLEXITY_API_KEY")))
print("Qdrant URL set:    ", bool(os.environ.get("QDRANT_URL")))
print("Qdrant key set:    ", bool(os.environ.get("QDRANT_API_KEY")))


<a name="embeddings"></a>
## 2. Embedding helper (Perplexity `pplx-embed-v1-0.6b`)

A tiny wrapper around `POST /v1/embeddings`. We ask for `base64_int8` (currently the only supported `encoding_format`), then decode to a `float32` list that Qdrant is happy to ingest.

The vector dimension for `pplx-embed-v1-0.6b` is **1024** — keep this in mind when you create the Qdrant collection.

We also stamp every request with an `X-Pplx-Integration` header so the API team can see traffic from this cookbook in their dashboards.


In [3]:
import base64
import httpx
import numpy as np

EMBEDDINGS_URL = "https://api.perplexity.ai/v1/embeddings"
EMBEDDING_MODEL = "pplx-embed-v1-0.6b"
EMBEDDING_DIM = 1024


def embed_texts(texts: list[str]) -> list[list[float]]:
    """Call Perplexity /v1/embeddings and return float32 vectors."""
    resp = httpx.post(
        EMBEDDINGS_URL,
        headers={
            "Authorization": f"Bearer {os.environ['PERPLEXITY_API_KEY']}",
            "Content-Type": "application/json",
            # See attribution header docs at docs.perplexity.ai
            "X-Pplx-Integration": "haystack/cookbook-live-research-agent",
        },
        json={
            "model": EMBEDDING_MODEL,
            "input": texts,
            "encoding_format": "base64_int8",
        },
        timeout=60.0,
    )
    resp.raise_for_status()
    out = []
    for item in resp.json()["data"]:
        raw = base64.b64decode(item["embedding"])
        vec = np.frombuffer(raw, dtype=np.int8).astype(np.float32)
        out.append(vec.tolist())
    return out


# Quick sanity check.
sample = embed_texts(["hello world", "retrieval augmented generation"])
print("returned", len(sample), "vectors of dim", len(sample[0]))


returned 2 vectors of dim 1024


<a name="doc-store"></a>
## 3. Qdrant document store

The Qdrant collection is created with `embedding_dim=1024` to match `pplx-embed-v1-0.6b`. We keep `recreate_index=True` here so re-running the notebook from scratch gives reproducible output — change to `False` once you want the index to persist between sessions.


In [4]:
from haystack.utils import Secret
from haystack_integrations.document_stores.qdrant import QdrantDocumentStore

document_store = QdrantDocumentStore(
    url=os.environ["QDRANT_URL"],
    api_key=Secret.from_token(os.environ["QDRANT_API_KEY"]),
    index="research_agent_demo",
    embedding_dim=EMBEDDING_DIM,
    similarity="cosine",
    recreate_index=True,   # set False once you want to keep the index across runs
    return_embedding=False,
)

print("Qdrant collection ready:", document_store.index)
print("Docs currently in index:", document_store.count_documents())


Qdrant collection ready: research_agent_demo


Docs currently in index: 0


<a name="seed"></a>
## 4. Seed the index

Two tight docs about *Haystack itself*. We intentionally leave out anything about the `perplexity-haystack` package so that one of the demo questions later forces the agent to call `web_search` + `ingest_url`.


In [5]:
from haystack import Document
from haystack.document_stores.types import DuplicatePolicy

seed_docs = [
    Document(
        content=(
            "Haystack is an open-source LLM framework by deepset. You compose "
            "components like retrievers, generators, and embedders into pipelines, "
            "and add tool-calling Agents on top."
        ),
        meta={"source": "https://haystack.deepset.ai/", "title": "Haystack overview"},
    ),
    Document(
        content=(
            "In Haystack 2.x, the Agent component takes a chat generator plus a "
            "list of Tool objects, loops over tool calls, and exits when the model "
            "emits a final answer (the 'text' exit condition)."
        ),
        meta={"source": "https://docs.haystack.deepset.ai/docs/agent", "title": "Haystack Agent component"},
    ),
]

# Attach embeddings before write — Qdrant needs the vectors up front.
seed_embeddings = embed_texts([d.content for d in seed_docs])
for doc, vec in zip(seed_docs, seed_embeddings):
    doc.embedding = vec

document_store.write_documents(seed_docs, policy=DuplicatePolicy.OVERWRITE)
print("Seeded docs in index:", document_store.count_documents())


/tmp/ipykernel_1731/3113823150.py:26: Warning: Mutating attribute 'embedding' on an instance of 'Document' can lead to unexpected behavior by affecting other parts of the pipeline that use the same dataclass instance. Use `dataclasses.replace(instance, embedding=new_value)` instead. See https://docs.haystack.deepset.ai/docs/custom-components#requirements for details.
  doc.embedding = vec


  0%|          | 0/2 [00:00<?, ?it/s]

100it [00:00, 431.47it/s]            

100it [00:00, 429.63it/s]

Seeded docs in index: 2


<a name="retriever"></a>
## 5. `retrieve_from_index` tool

Embed the query with the same model used at indexing time, then pull the top-k from Qdrant. The tool returns a compact JSON payload — title, source URL, snippet, score — because the agent has to fit it into its context window.


In [6]:
import json


In [7]:
from haystack_integrations.components.retrievers.qdrant import QdrantEmbeddingRetriever

retriever = QdrantEmbeddingRetriever(document_store=document_store, top_k=4)


def retrieve_from_index(query: str, top_k: int = 4) -> dict:
    query_emb = embed_texts([query])[0]
    hits = retriever.run(query_embedding=query_emb, top_k=top_k)["documents"]
    return {
        "hits": [
            {
                "title": d.meta.get("title", ""),
                "source": d.meta.get("source", ""),
                "snippet": d.content[:400],
                "score": round(d.score, 4),
            }
            for d in hits
        ]
    }


# Smoke test — should retrieve the seed docs about Haystack.
preview = retrieve_from_index("What is Haystack?", top_k=2)
print(json.dumps(preview, indent=2)[:800])


{
  "hits": [
    {
      "title": "Haystack overview",
      "source": "https://haystack.deepset.ai/",
      "snippet": "Haystack is an open-source LLM framework by deepset. You compose components like retrievers, generators, and embedders into pipelines, and add tool-calling Agents on top.",
      "score": 0.6988
    },
    {
      "title": "Haystack Agent component",
      "source": "https://docs.haystack.deepset.ai/docs/agent",
      "snippet": "In Haystack 2.x, the Agent component takes a chat generator plus a list of Tool objects, loops over tool calls, and exits when the model emits a final answer (the 'text' exit condition).",
      "score": 0.4161
    }
  ]
}


<a name="web-search"></a>
## 6. `web_search` tool (Perplexity Search API)

`PerplexityWebSearch` hits `POST /search` and gives back already-ranked, already-cleaned results plus the list of source URLs. No SERP scraper, no extra fetcher in front of the model.


In [8]:
from haystack_integrations.components.websearch.perplexity import PerplexityWebSearch

websearch = PerplexityWebSearch(top_k=5)


def web_search(query: str, top_k: int = 5) -> dict:
    r = websearch.run(query=query, search_params={"max_results": top_k})
    return {
        "results": [
            {
                "title": d.meta.get("title", ""),
                "url": d.meta.get("url") or d.meta.get("link", ""),
                "snippet": d.content[:400],
            }
            for d in r["documents"]
        ],
        "links": r["links"],
    }


preview = web_search("perplexity haystack integration overview", top_k=3)
print(json.dumps(preview, indent=2)[:900])


{
  "results": [
    {
      "title": "Perplexity with Haystack",
      "url": "https://docs.perplexity.ai/docs/getting-started/integrations/haystack",
      "snippet": "## \u200b Overview\nThe `perplexity-haystack` package provides Haystack components for Perplexity\u2019s Agent API, Embeddings API, and grounded Search API, so you can build retrieval-augmented and agentic pipelines that combine chat, embeddings, and live web search.\n**Haystack** is an open-source Python framework by deepset for building production-ready LLM applications, including RAG pipelines and agentic "
    },
    {
      "title": "Perplexity | Haystack - deepset AI",
      "url": "https://haystack.deepset.ai/integrations/perplexity",
      "snippet": "# Integration: Perplexity\nUse the Perplexity Agent API, Embeddings API, and grounded Search API in Haystack pipelines.\n...\n## Overview\nThe `perplexity-haystack`


<a name="ingest"></a>
## 7. `ingest_url` tool

Fetch a URL with `trafilatura` (with an `httpx` fallback for sites that 403 the default UA), pull the main article text, chunk it with Haystack's `DocumentSplitter`, embed the chunks, and write them to Qdrant.

Once a page is ingested, future `retrieve_from_index` calls can hit it without paying for another web request.


In [9]:
import trafilatura
from haystack.components.preprocessors import DocumentSplitter

splitter = DocumentSplitter(split_by="word", split_length=200, split_overlap=20)
splitter.warm_up()


def ingest_url(url: str, title: str | None = None) -> dict:
    # trafilatura's own fetcher is fine on clean pages; some sites block the
    # default UA, so fall back to httpx with a browser-ish UA.
    raw = trafilatura.fetch_url(url)
    if not raw:
        try:
            raw = httpx.get(
                url, timeout=20, follow_redirects=True,
                headers={"User-Agent": "Mozilla/5.0 (Haystack cookbook demo)"},
            ).text
        except Exception as e:
            return {"ok": False, "reason": f"fetch error: {e}", "url": url}
    text = trafilatura.extract(raw)
    if not text or len(text) < 200:
        return {"ok": False, "reason": "could not extract enough content", "url": url}

    doc = Document(content=text, meta={"source": url, "title": title or url})
    chunks = splitter.run(documents=[doc])["documents"]

    chunk_vecs = embed_texts([c.content for c in chunks])
    for c, v in zip(chunks, chunk_vecs):
        c.embedding = v

    document_store.write_documents(chunks, policy=DuplicatePolicy.OVERWRITE)
    return {
        "ok": True,
        "url": url,
        "chunks_indexed": len(chunks),
        "chars_extracted": len(text),
        "total_docs_in_index": document_store.count_documents(),
    }


# Demo: ingest the Perplexity integrations landing page so the agent
# can answer questions about perplexity-haystack later from the index.
print(json.dumps(
    ingest_url("https://haystack.deepset.ai/integrations/perplexity",
               title="Perplexity Haystack integration"),
    indent=2,
))


/tmp/ipykernel_1731/3725313797.py:29: Warning: Mutating attribute 'embedding' on an instance of 'Document' can lead to unexpected behavior by affecting other parts of the pipeline that use the same dataclass instance. Use `dataclasses.replace(instance, embedding=new_value)` instead. See https://docs.haystack.deepset.ai/docs/custom-components#requirements for details.
  c.embedding = v


  0%|          | 0/2 [00:00<?, ?it/s]

100it [00:00, 430.75it/s]            

100it [00:00, 429.83it/s]

{
  "ok": true,
  "url": "https://haystack.deepset.ai/integrations/perplexity",
  "chunks_indexed": 2,
  "chars_extracted": 3061,
  "total_docs_in_index": 4
}


<a name="agent"></a>
## 8. Agent (Perplexity Agent API)

`PerplexityChatGenerator` defaults to `openai/gpt-5.4` via the Agent API; you can swap to any other model the Agent API exposes (Anthropic, Gemini, Perplexity Sonar, etc.).

The system prompt forces the order **retrieve → web_search → ingest_url → retrieve again → answer** so we get a clean demo of the loop. In production you can loosen it.


In [10]:
from haystack.tools import Tool
from haystack.components.agents import Agent
from haystack.dataclasses import ChatMessage
from haystack_integrations.components.generators.perplexity import PerplexityChatGenerator

retrieve_tool = Tool(
    name="retrieve_from_index",
    description=(
        "Search the local Qdrant knowledge base (Perplexity embeddings). "
        "Use this FIRST for any question that might be in the index."
    ),
    parameters={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "top_k": {"type": "integer", "default": 4, "minimum": 1, "maximum": 10},
        },
        "required": ["query"],
    },
    function=retrieve_from_index,
)

web_search_tool = Tool(
    name="web_search",
    description=(
        "Search the live web with the Perplexity Search API. Use when "
        "retrieve_from_index returns nothing useful, or when you need current information."
    ),
    parameters={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "top_k": {"type": "integer", "default": 5, "minimum": 1, "maximum": 20},
        },
        "required": ["query"],
    },
    function=web_search,
)

ingest_tool = Tool(
    name="ingest_url",
    description=(
        "Fetch a URL, embed it with Perplexity embeddings, and add it to the "
        "Qdrant index for reuse by future retrieve_from_index calls. Use after "
        "web_search when you find an authoritative source."
    ),
    parameters={
        "type": "object",
        "properties": {
            "url": {"type": "string"},
            "title": {"type": "string"},
        },
        "required": ["url"],
    },
    function=ingest_url,
)

chat = PerplexityChatGenerator(model="openai/gpt-5.4")

agent = Agent(
    chat_generator=chat,
    tools=[retrieve_tool, web_search_tool, ingest_tool],
    system_prompt=(
        "You are a research agent. For every user question:\n"
        "1. Call retrieve_from_index first to see if the local knowledge base already has the answer.\n"
        "2. If the retrieved snippets don't cover the question, call web_search.\n"
        "3. Whenever web_search returns a URL whose contents you actually need to answer the question, "
        "call ingest_url on that URL FIRST so future retrieve_from_index calls can find it. "
        "Then call retrieve_from_index again to read the chunks you just ingested.\n"
        "4. Write a concise answer. Cite every fact with the source URL it came from using inline markdown links."
    ),
    exit_conditions=["text"],
    max_agent_steps=10,
)
agent.warm_up()
print("Agent ready with tools:", [t.name for t in agent.tools])


Agent ready with tools: ['retrieve_from_index', 'web_search', 'ingest_url']


<a name="run-demo"></a>
## 9. Run the agent on three questions

* **Q1** is covered by the seed docs — should be one `retrieve_from_index` call, then an answer.
* **Q2** isn't covered — the agent should fall through to `web_search`, pick a URL, `ingest_url` it, and retrieve again before answering.
* **Q3** asks something that *should* now be in the index thanks to Q2's ingest, so the agent can stay local.

For each run we print the message timeline (role + tool calls), then the final answer.


In [11]:
QUESTIONS = [
    # Q1 — answerable from the seed docs alone.
    "What is the Haystack Agent component, briefly?",
    # Q2 — not in the index. Forces web_search + ingest_url.
    "What does the perplexity-haystack integration package provide, according to the official Haystack integrations page?",
    # Q3 — should now hit the page Q2 ingested.
    "Which Perplexity APIs does the perplexity-haystack package wrap, and which Haystack component classes does it ship?",
]


def run_question(q: str) -> None:
    print("=" * 80)
    print("Q:", q)
    result = agent.run(messages=[ChatMessage.from_user(q)])
    msgs = result["messages"]
    print(f"messages: {len(msgs)}")
    for i, m in enumerate(msgs):
        tool_calls = getattr(m, "tool_calls", []) or []
        tool_results = getattr(m, "tool_call_results", []) or []
        print(
            f"  [{i}] role={m.role.value} "
            f"tool_calls={[t.tool_name for t in tool_calls]} "
            f"results={len(tool_results)} text_len={len(m.text or '')}"
        )
    print("\nFINAL ANSWER:\n" + (msgs[-1].text or ""))
    print("\nDocs in index now:", document_store.count_documents())


for q in QUESTIONS:
    run_question(q)
    print()


Q: What is the Haystack Agent component, briefly?


/tmp/ipykernel_1731/3725313797.py:29: Warning: Mutating attribute 'embedding' on an instance of 'Document' can lead to unexpected behavior by affecting other parts of the pipeline that use the same dataclass instance. Use `dataclasses.replace(instance, embedding=new_value)` instead. See https://docs.haystack.deepset.ai/docs/custom-components#requirements for details.
  c.embedding = v


  0%|          | 0/12 [00:00<?, ?it/s]

100it [00:00, 283.69it/s]             

100it [00:00, 283.36it/s]

...

  0%|          | 0/2 [00:00<?, ?it/s]

100it [00:00, 430.24it/s]            

100it [00:00, 429.36it/s]

messages: 15
  [0] role=system tool_calls=[] results=0 text_len=571
  [1] role=user tool_calls=[] results=0 text_len=115
  [2] role=assistant tool_calls=['retrieve_from_index'] results=0 text_len=0
  [3] role=tool tool_calls=[] results=1 text_len=0
  [4] role=assistant tool_calls=['web_search'] results=0 text_len=0
  [5] role=tool tool_calls=[] results=1 text_len=0
  [6] role=assistant tool_calls=['ingest_url'] results=0 text_len=0
  [7] role=tool tool_calls=[] results=1 text_len=0
  [8] role=assistant tool_calls=['ingest_url'] results=0 text_len=0
  [9] role=tool tool_calls=[] results=1 text_len=0
  [10] role=assistant tool_calls=['ingest_url'] results=0 text_len=0
  [11] role=tool tool_calls=[] results=1 text_len=0
  [12] role=assistant tool_calls=['retrieve_from_index'] results=0 text_len=0
  [13] role=tool tool_calls=[] results=1 text_len=0
  [14] role=assistant tool_calls=[] results=0 text_len=445

FINAL ANSWER:
`perplexity-haystack` wraps Perplexity’s **chat-completions API** and


Docs in index now: 27



<a name="wrap-up"></a>
## 10. Where to go next

* Swap `recreate_index=True` to `False` and let the index accumulate across sessions — every web answer the agent gives stays available offline.
* Bring in `PerplexityContextualizedEmbedder` (`/v1/contextualizedembeddings`) instead of the static doc embedder if your corpus has heavy section structure or codebases.
* Add a `cite_index` tool that returns the embedded chunk IDs alongside the answer, so downstream UIs can hyperlink back to the source.
* Try a different reasoning model — `chat = PerplexityChatGenerator(model="anthropic/claude-sonnet-4-5")` works via the Agent API too.

### References
* Perplexity Agent API — <https://docs.perplexity.ai/api-reference/agent-post>
* Perplexity Search API — <https://docs.perplexity.ai/api-reference/search-post>
* Perplexity Embeddings API — <https://docs.perplexity.ai/api-reference/embeddings-post>
* Haystack Perplexity integration — <https://haystack.deepset.ai/integrations/perplexity>
* Qdrant document store — <https://haystack.deepset.ai/integrations/qdrant-document-store>
